In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt 
import plotly.express as px
import random

In [2]:
data = pd.read_csv('Pokemon.csv')

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   #           800 non-null    int64 
 1   Name        800 non-null    object
 2   Type 1      800 non-null    object
 3   Type 2      414 non-null    object
 4   Total       800 non-null    int64 
 5   HP          800 non-null    int64 
 6   Attack      800 non-null    int64 
 7   Defense     800 non-null    int64 
 8   Sp. Atk     800 non-null    int64 
 9   Sp. Def     800 non-null    int64 
 10  Speed       800 non-null    int64 
 11  Generation  800 non-null    int64 
 12  Legendary   800 non-null    bool  
dtypes: bool(1), int64(9), object(3)
memory usage: 75.9+ KB


In [ ]:
# data['Power'] = data['Type 2'].fillna(data['Type 1'])

# pre_processed_data = data.drop(columns=['Type 1', 'Type 2'], inplace=False)


In [9]:
def determine_power(row):
    t1 = str(row['Type 1']).upper()
    t2 = row['Type 2']
    
    # If Type 2 is empty or NaN, Power is just Type 1
    if pd.isna(t2) or str(t2).strip() == "":
        return t1
    else:
        # Otherwise, combine them into a Hybrid Power
        return f"{t1}-{str(t2).upper()}"

# 3. Apply the function to create the new column
data['Power'] = data.apply(determine_power, axis=1)

pre_processed_data = data.drop(columns=['Type 1', 'Type 2'], inplace=False)

In [11]:

stat_columns = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']
num_samples_per_pokemon = 100 

augmented_data = []

for index, row in pre_processed_data.iterrows():

    original_row = row.to_dict()
    original_row['Is_Augmented'] = 0 
    augmented_data.append(original_row)
    
    for _ in range(num_samples_per_pokemon - 1): 
        
        new_row = row.to_dict()
        
        for stat in stat_columns:

            base_val = new_row[stat]
            noise_val = np.random.normal(loc=base_val, scale=base_val * 0.05)
            new_row[stat] = max(1, int(round(noise_val)))

            
            
        new_row['Total'] = sum(new_row[stat] for stat in stat_columns)
        new_row['Is_Augmented'] = 1
        augmented_data.append(new_row)


augmented_df = pd.DataFrame(augmented_data)

augmented_df['Legendary'] = augmented_df['Legendary'].astype(int)

augmented_df.to_csv('augmented_pokemon.csv', index=False)

print(f"Original shape: {pre_processed_data.shape}")
print(f"Augmented shape: {augmented_df.shape}")

Original shape: (800, 12)
Augmented shape: (80000, 13)


In [19]:
name = random.choice(data["Name"].unique())
print(f"Randomly selected Pokémon: {name}")

print(data[data["Name"]==name])

augmented_df[augmented_df["Name"]==name].sample(20)

Randomly selected Pokémon: Axew
       #  Name  Type 1 Type 2  Total  HP  Attack  Defense  Sp. Atk  Sp. Def  \
671  610  Axew  Dragon    NaN    320  46      87       60       30       40   

     Speed  Generation  Legendary   Power  
671     57           5      False  DRAGON  


,#,Name,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary,Power,Is_Augmented
67197,610,Axew,314,47,82,58,31,43,53,5,0,DRAGON,1
67159,610,Axew,319,48,87,57,31,37,59,5,0,DRAGON,1
67182,610,Axew,310,47,85,56,28,39,55,5,0,DRAGON,1
67137,610,Axew,306,53,75,55,32,38,53,5,0,DRAGON,1
67180,610,Axew,309,45,83,63,29,37,52,5,0,DRAGON,1
67141,610,Axew,322,49,90,54,30,38,61,5,0,DRAGON,1
67184,610,Axew,325,46,88,61,31,39,60,5,0,DRAGON,1
67143,610,Axew,318,48,85,61,28,38,58,5,0,DRAGON,1
67103,610,Axew,319,46,81,58,30,40,64,5,0,DRAGON,1
67153,610,Axew,315,44,89,63,29,37,53,5,0,DRAGON,1


In [24]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

X1_without_power = augmented_df[['HP' , 'Attack' , 'Defense' , 'Sp. Atk' , 'Sp. Def' , 'Speed']]
X2_with_power = augmented_df[['HP' , 'Attack' , 'Defense' , 'Sp. Atk' , 'Sp. Def' , 'Speed' ,'Legendary','Generation', 'Power']]

X2_with_power = pd.get_dummies(X2_with_power, columns=['Power'] , drop_first=True)

y_power_label = augmented_df['Power']
y_name_label = augmented_df['Name']
y_legendary_label = augmented_df['Legendary']
y_generation_label = augmented_df['Generation']


power_encoder = LabelEncoder()
name_encoder = LabelEncoder()
legendary_encoder = LabelEncoder()
generation_encoder = LabelEncoder()


y_power_encoded = power_encoder.fit_transform(y_power_label)
y_name_encoded = name_encoder.fit_transform(y_name_label)
y_legendary_encoded = legendary_encoder.fit_transform(y_legendary_label)
y_generation_encoded = generation_encoder.fit_transform(y_generation_label)



X_train_topower  , X_test_topower, y_train_power, y_test_power = train_test_split(X1_without_power, y_power_encoded, test_size=0.1, random_state=42 , stratify=y_power_encoded , shuffle=True)

X_train_toname, X_test_toname, y_train_name, y_test_name = train_test_split(X2_with_power, y_name_encoded, test_size=0.1, random_state=42 , stratify=y_name_encoded , shuffle=True)

X_train_tolegendary, X_test_tolegendary, y_train_legendary, y_test_legendary = train_test_split(X1_without_power, y_legendary_encoded, test_size=0.1, random_state=42 , stratify=y_legendary_encoded , shuffle=True)

X_train_togeneration, X_test_togeneration, y_train_generation, y_test_generation = train_test_split(X1_without_power, y_generation_encoded, test_size=0.1, random_state=42 , stratify=y_generation_encoded , shuffle=True)



In [22]:
training_columns = X2_with_power.columns.tolist()
print(f"Model 2 expects {len(training_columns)} columns:")
print(training_columns)

Model 2 expects 161 columns:
['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Legendary', 'Generation', 'Power_BUG-ELECTRIC', 'Power_BUG-FIGHTING', 'Power_BUG-FIRE', 'Power_BUG-FLYING', 'Power_BUG-GHOST', 'Power_BUG-GRASS', 'Power_BUG-GROUND', 'Power_BUG-POISON', 'Power_BUG-ROCK', 'Power_BUG-STEEL', 'Power_BUG-WATER', 'Power_DARK', 'Power_DARK-DRAGON', 'Power_DARK-FIGHTING', 'Power_DARK-FIRE', 'Power_DARK-FLYING', 'Power_DARK-GHOST', 'Power_DARK-ICE', 'Power_DARK-PSYCHIC', 'Power_DARK-STEEL', 'Power_DRAGON', 'Power_DRAGON-ELECTRIC', 'Power_DRAGON-FAIRY', 'Power_DRAGON-FIRE', 'Power_DRAGON-FLYING', 'Power_DRAGON-GROUND', 'Power_DRAGON-ICE', 'Power_DRAGON-PSYCHIC', 'Power_ELECTRIC', 'Power_ELECTRIC-DRAGON', 'Power_ELECTRIC-FAIRY', 'Power_ELECTRIC-FIRE', 'Power_ELECTRIC-FLYING', 'Power_ELECTRIC-GHOST', 'Power_ELECTRIC-GRASS', 'Power_ELECTRIC-ICE', 'Power_ELECTRIC-NORMAL', 'Power_ELECTRIC-STEEL', 'Power_ELECTRIC-WATER', 'Power_FAIRY', 'Power_FAIRY-FLYING', 'Power_FIGHTING', 'Pow

In [27]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score , precision_score , recall_score , f1_score



model_power_predictor = RandomForestClassifier()

model_power_predictor.fit(X_train_topower, y_train_power)
y_pred_power = model_power_predictor.predict(X_test_topower)


accuracy = accuracy_score(y_test_power, y_pred_power)
precision = precision_score(y_test_power, y_pred_power, average='weighted')
recall = recall_score(y_test_power, y_pred_power, average='weighted')
f1 = f1_score(y_test_power, y_pred_power, average='weighted')


print("Model 1 (Predicting Power):")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

print("=" * 30)


model_legendary_predictor = RandomForestClassifier()

model_legendary_predictor.fit(X_train_tolegendary, y_train_legendary)
y_pred_legendary = model_legendary_predictor.predict(X_test_tolegendary)


accuracy = accuracy_score(y_test_legendary, y_pred_legendary)
precision = precision_score(y_test_legendary, y_pred_legendary, average='weighted')
recall = recall_score(y_test_legendary, y_pred_legendary, average='weighted')
f1 = f1_score(y_test_legendary, y_pred_legendary, average='weighted')


print("Model 2 (Predicting Legendary):")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
# print(f"F1 Score: {f1:.4f}")

print("=" * 30)

model_generation_predictor = RandomForestClassifier()

model_generation_predictor.fit(X_train_togeneration, y_train_generation)
y_pred_generation = model_generation_predictor.predict(X_test_togeneration)


accuracy = accuracy_score(y_test_generation, y_pred_generation)
precision = precision_score(y_test_generation, y_pred_generation, average='weighted')
recall = recall_score(y_test_generation, y_pred_generation, average='weighted')
f1 = f1_score(y_test_generation, y_pred_generation, average='weighted')


print("Model 3 (Predicting Generation):")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

print("=" * 30)

model_name_predictor = RandomForestClassifier()

model_name_predictor.fit(X_train_toname, y_train_name)
y_pred_name = model_name_predictor.predict(X_test_toname)


accuracy2 = accuracy_score(y_test_name, y_pred_name)
precision2 = precision_score(y_test_name, y_pred_name, average='weighted')
recall2 = recall_score(y_test_name, y_pred_name, average='weighted')
f1_2 = f1_score(y_test_name, y_pred_name, average='weighted')

print("Model 4 (Predicting Name):")
print(f"Accuracy: {accuracy2:.4f}")
print(f"Precision: {precision2:.4f}")
print(f"Recall: {recall2:.4f}")
print(f"F1 Score: {f1_2:.4f}")



Model 1 (Predicting Power):
Accuracy: 0.9441
Precision: 0.9443
Recall: 0.9441
F1 Score: 0.9438
Model 2 (Predicting Legendary):
Accuracy: 0.9908
Precision: 0.9907
Recall: 0.9908
Model 3 (Predicting Generation):
Accuracy: 0.9581
Precision: 0.9581
Recall: 0.9581
F1 Score: 0.9581
Model 4 (Predicting Name):
Accuracy: 0.9944
Precision: 0.9945
Recall: 0.9944
F1 Score: 0.9943


Model 1 (Predicting Power):
- Accuracy: 0.9539
- Precision: 0.9539
- Recall: 0.9539
- F1 Score: 0.9538

==============================

Model 2 (Predicting Legendary):
- Accuracy: 0.9915
- Precision: 0.9914
- Recall: 0.9915
- F1 Score: 0.9915

==============================

Model 3 (Predicting Generation):
- Accuracy: 0.9599
- Precision: 0.9599
- Recall: 0.9599
- F1 Score: 0.9599

==============================

Model 4 (Predicting Name):
- Accuracy: 0.9935
- Precision: 0.9935
- Recall: 0.9935
- F1 Score: 0.9934

In [26]:
from sklearn.ensemble import RandomForestClassifier


PowerPredictor = RandomForestClassifier()
PowerPredictor.fit(X1_without_power, y_power_encoded)


LegendaryPredictor = RandomForestClassifier()
LegendaryPredictor.fit(X1_without_power, y_legendary_encoded)


GenerationPredictor = RandomForestClassifier()
GenerationPredictor.fit(X1_without_power, y_generation_encoded)


NamePredictor = RandomForestClassifier()
NamePredictor.fit(X2_with_power, y_name_encoded)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [ ]:
# import os
# import joblib

# predict_folder = 'models\predictors'
# encoder_folder = 'models\encoders'

# if not os.path.exists(predict_folder):
#     os.makedirs(predict_folder)

# if not os.path.exists(encoder_folder):
#     os.makedirs(encoder_folder)

# joblib.dump(PowerPredictor, os.path.join(predict_folder, 'PowerPredictor.pkl') , compress=3)
# print(f"Model saved as {'PowerPredictor.pkl'}")
# joblib.dump(NamePredictor, os.path.join(predict_folder, 'NamePredictor.pkl') ,  compress=3)
# print(f"Model saved as {'NamePredictor.pkl'}")
# joblib.dump(LegendaryPredictor, os.path.join(predict_folder, 'LegendaryPredictor.pkl') ,  compress=3)
# print(f"Model saved as {'LegendaryPredictor.pkl'}")
# joblib.dump(GenerationPredictor, os.path.join(predict_folder, 'GenerationPredictor.pkl') ,  compress=3)
# print(f"Model saved as {'GenerationPredictor.pkl'}")


# joblib.dump(power_encoder, os.path.join(encoder_folder, 'power_encoder.pkl') , compress=3)
# print(f"Model saved as {'power_encoder.pkl'}")
# joblib.dump(name_encoder, os.path.join(encoder_folder, 'name_encoder.pkl') ,  compress=3)
# print(f"Model saved as {'name_encoder.pkl'}")
# joblib.dump(legendary_encoder, os.path.join(encoder_folder, 'legendary_encoder.pkl') ,  compress=3)
# print(f"Model saved as {'legendary_encoder.pkl'}")
# joblib.dump(generation_encoder, os.path.join(encoder_folder, 'generation_encoder.pkl') ,  compress=3)
# print(f"Model saved as {'generation_encoder.pkl'}")


In [28]:
import requests
from IPython.display import Image, display, HTML

# Build name -> Pokédex ID map directly from your CSV
name_to_id = dict(zip(data['Name'], data['#']))

print(f"Lookup table ready: {len(name_to_id)} Pokémon")

def show_pokemon(predicted_name):
    pokedex_id = name_to_id.get(predicted_name)
    if pokedex_id is None:
        print(f"Not found: {predicted_name}")
        return
    
    url = f"https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/other/official-artwork/{pokedex_id}.png"
    display(HTML(f'<img src="{url}" width="200px"/>'))

Lookup table ready: 800 Pokémon


In [ ]:
# import joblib

# PowerPredictor = joblib.load('models/predictors/PowerPredictor.pkl')
# LegendaryPredictor = joblib.load('models/predictors/LegendaryPredictor.pkl')
# GenerationPredictor = joblib.load('models/predictors/GenerationPredictor.pkl')
# NamePredictor = joblib.load('models/predictors/NamePredictor.pkl')

# power_encoder = joblib.load('models/encoders/power_encoder.pkl')
# name_encoder = joblib.load('models/encoders/name_encoder.pkl')
# legendary_encoder = joblib.load('models/encoders/legendary_encoder.pkl')
# generation_encoder = joblib.load('models/encoders/generation_encoder.pkl')

In [ ]:
health_point = random.randint(augmented_df['HP'].min(), augmented_df['HP'].max())
print(f"Randomly selected HP value: {health_point}")

attack = random.randint(augmented_df['Attack'].min(), augmented_df['Attack'].max())
print(f"Randomly selected Attack value: {attack}")

defense = random.randint(augmented_df['Defense'].min(), augmented_df['Defense'].max())
print(f"Randomly selected Defense value: {defense}")

sp_atk = random.randint(augmented_df['Sp. Atk'].min(), augmented_df['Sp. Atk'].max())
print(f"Randomly selected Sp. Atk value: {sp_atk}")

sp_def = random.randint(augmented_df['Sp. Def'].min(), augmented_df['Sp. Def'].max())
print(f"Randomly selected Sp. Def value: {sp_def}")

speed = random.randint(augmented_df['Speed'].min(), augmented_df['Speed'].max())
print(f"Randomly selected Speed value: {speed}")



new_pokemon = pd.DataFrame({
    'HP': [health_point],
    'Attack': [attack],
    'Defense': [defense],
    'Sp. Atk': [sp_atk],
    'Sp. Def': [sp_def],
    'Speed': [speed]
    })



# new_pokemon = pd.DataFrame({
#     'HP': [78],
#     'Attack': [84],
#     'Defense': [78],
#     'Sp. Atk': [109],
#     'Sp. Def': [85],
#     'Speed': [100],
#     })


predicted_power_encoded = PowerPredictor.predict(new_pokemon)[0]
predicted_power = power_encoder.inverse_transform([predicted_power_encoded])[0]

print(f"Predicted Power: {predicted_power}")

predicted_legendary_encoded = LegendaryPredictor.predict(new_pokemon)[0]
predicted_legendary = legendary_encoder.inverse_transform([predicted_legendary_encoded])[0]

predicted_generation_encoded = GenerationPredictor.predict(new_pokemon)[0]
predicted_generation = generation_encoder.inverse_transform([predicted_generation_encoded])[0]

new_pokemon['Legendary'] = predicted_legendary
new_pokemon['Generation'] = predicted_generation
new_pokemon[[f'Power_{predicted_power}']] = 1


new_pokemon = new_pokemon.reindex(columns=training_columns, fill_value=0)


predicted_name_encoded = NamePredictor.predict(new_pokemon)[0]
predicted_name = name_encoder.inverse_transform([predicted_name_encoded])[0]

print(f"Predicted Pokémon: {predicted_name}")
print(f"Pokemon Power is: {data[data['Name'] == predicted_name]['Power'].iloc[0]}")

show_pokemon(predicted_name)

new_pokemon[[f'Power_{predicted_power}']]

Randomly selected HP value: 93
Randomly selected Attack value: 25
Randomly selected Defense value: 68
Randomly selected Sp. Atk value: 70
Randomly selected Sp. Def value: 245
Randomly selected Speed value: 188
Predicted Power: WATER-FLYING
Predicted Pokémon: Mantine
Pokemon Power is: WATER-FLYING


,Power_WATER-FLYING
0,1


In [43]:
from tqdm import tqdm

def PokemonPredictor(health_point = None , attack = None , defense = None , sp_atk = None , sp_def = None , speed = None) : 

    if health_point is None:
        health_point = random.randint(augmented_df['HP'].min(), augmented_df['HP'].max())

    if attack is None:
        attack = random.randint(augmented_df['Attack'].min(), augmented_df['Attack'].max())

    if defense is None:
        defense = random.randint(augmented_df['Defense'].min(), augmented_df['Defense'].max())

    if sp_atk is None:
        sp_atk = random.randint(augmented_df['Sp. Atk'].min(), augmented_df['Sp. Atk'].max())

    if sp_def is None:
        sp_def = random.randint(augmented_df['Sp. Def'].min(), augmented_df['Sp. Def'].max())

    if speed is None:
        speed = random.randint(augmented_df['Speed'].min(), augmented_df['Speed'].max())



    new_pokemon = pd.DataFrame({
        'HP': [health_point],
        'Attack': [attack],
        'Defense': [defense],
        'Sp. Atk': [sp_atk],
        'Sp. Def': [sp_def],
        'Speed': [speed]
        })
    

    predicted_power_encoded = PowerPredictor.predict(new_pokemon)[0]
    predicted_power = power_encoder.inverse_transform([predicted_power_encoded])[0]

    predicted_legendary_encoded = LegendaryPredictor.predict(new_pokemon)[0]
    predicted_legendary = legendary_encoder.inverse_transform([predicted_legendary_encoded])[0]

    predicted_generation_encoded = GenerationPredictor.predict(new_pokemon)[0]
    predicted_generation = generation_encoder.inverse_transform([predicted_generation_encoded])[0]

    new_pokemon['Legendary'] = predicted_legendary
    new_pokemon['Generation'] = predicted_generation
    new_pokemon[[f'Power_{predicted_power}']] = 1


    new_pokemon = new_pokemon.reindex(columns=training_columns, fill_value=0)


    predicted_name_encoded = NamePredictor.predict(new_pokemon)[0]
    predicted_name = name_encoder.inverse_transform([predicted_name_encoded])[0]

    
    return predicted_name 


# team = []

# for _ in tqdm(range(10000), desc="Generating team"):

#     pokemon = PokemonPredictor()
#     team.append(pokemon)


# team = set(team)

# print(f"after 10000 iterations, we got {len(team)} unique pokemons in the team")



PredData = data.copy()

PredData['Predicted Name'] = PredData.apply(lambda row: PokemonPredictor(
    health_point=row['HP'],
    attack=row['Attack'],
    defense=row['Defense'],
    sp_atk=row['Sp. Atk'],
    sp_def=row['Sp. Def'],
    speed=row['Speed']
), axis=1)








    

In [44]:
PredData['Right Prediction'] = PredData['Name'] == PredData['Predicted Name']

print(PredData['Right Prediction'].value_counts())

PredData[PredData['Right Prediction'] == False][['Name', 'Predicted Name', 'HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']].head(30)

Right Prediction
True     778
False     22
Name: count, dtype: int64


,Name,Predicted Name,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed
4,Charmander,Cyndaquil,39,52,43,60,50,65
165,Mew,Victini,100,100,100,100,100,100
170,Quilava,Charmeleon,58,64,58,80,65,80
171,Typhlosion,Charizard,78,84,78,109,85,100
271,Celebi,Victini,100,100,100,100,100,100
289,Silcoon,Cascoon,50,35,55,25,25,15
427,Jirachi,Victini,100,100,100,100,100,100
532,RotomHeat Rotom,RotomFrost Rotom,50,65,107,105,107,86
533,RotomWash Rotom,RotomFrost Rotom,50,65,107,105,107,86
535,RotomFan Rotom,RotomFrost Rotom,50,65,107,105,107,86
